In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_squared_error


# 1. Load dataset
dataset = pd.read_csv("Preplacementdata.csv")

dataset=dataset.drop('sl_no',axis=1)

dataset
# 2. Convert categorical data into numerical data
df2 = pd.get_dummies(dataset, drop_first=True, dtype=int)


# 3. Separate independent and dependent variables
indep_X = df2.drop('salary', axis=1)
dep_Y = df2['salary']


# 4. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    indep_X, dep_Y,
    test_size=0.25,
    random_state=0
)


# 5. Feature Scaling
sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


# 6. Create different algorithms for RFE
rfe_models = {
    'Linear Regression': LinearRegression(),

    'SVM Linear': SVR(kernel='linear'),

    'Decision Tree': DecisionTreeRegressor(
        random_state=0
    ),

    'Random Forest': RandomForestRegressor(
        n_estimators=10,
        random_state=0
    )
}


# 7. Create different regression algorithms for evaluation
regression_models = {
    'Linear Regression': LinearRegression(),

    'SVM Linear': SVR(kernel='linear'),

    'SVM Non-Linear': SVR(kernel='rbf'),

    'Decision Tree': DecisionTreeRegressor(
        random_state=0
    ),

    'Random Forest': RandomForestRegressor(
        n_estimators=10,
        random_state=0
    )
}


# 8. Create lists to store results
all_results = []
selected_feature_results = []


# 9. Apply RFE using different algorithms
for rfe_name, rfe_model in rfe_models.items():

    print("\n===================================")
    print("RFE Algorithm:", rfe_name)
    print("===================================")

    # Create RFE
    rfe = RFE(
        estimator=rfe_model,
        n_features_to_select=3
    )

    # Fit RFE using training data
    rfe.fit(X_train, y_train)

    # Get selected features
    selected_features = indep_X.columns[rfe.support_]

    print("\nSelected Features:")
    print(list(selected_features))

    # Store selected feature information
    selected_feature_results.append({
        'RFE Algorithm': rfe_name,
        'Selected Features': list(selected_features)
    })

    # Transform training and testing data
    X_train_selected = rfe.transform(X_train)
    X_test_selected = rfe.transform(X_test)


    # 10. Evaluate all regression algorithms
    for model_name, model in regression_models.items():

        # Train model
        model.fit(X_train_selected, y_train)

        # Predict test data
        y_pred = model.predict(X_test_selected)

        # Calculate R2 score
        r2 = r2_score(y_test, y_pred)

        # Calculate MSE
        mse = mean_squared_error(y_test, y_pred)

        # Store results
        all_results.append({
            'RFE Algorithm': rfe_name,
            'Regression Algorithm': model_name,
            'R2 Score': r2,
            'MSE': mse
        })


# 11. Display selected features
feature_df = pd.DataFrame(selected_feature_results)

print("\nSELECTED FEATURES COMPARISON")
print(feature_df.to_string(index=False))


# 12. Display complete comparison
result_df = pd.DataFrame(all_results)

print("\nALL ALGORITHM COMPARISON")
print(result_df)


# 13. Compare R2 scores in a table
r2_comparison = result_df.pivot(
    index='RFE Algorithm',
    columns='Regression Algorithm',
    values='R2 Score'
)

print("\nR2 SCORE COMPARISON")
print(r2_comparison)


# 14. Compare MSE scores in a table
mse_comparison = result_df.pivot(
    index='RFE Algorithm',
    columns='Regression Algorithm',
    values='MSE'
)

print("\nMSE COMPARISON")
print(mse_comparison)


# 15. Find the best combination based on R2
best_result = result_df.loc[
    result_df['R2 Score'].idxmax()
]

print("\nBEST RFE AND REGRESSION COMBINATION")
print(best_result)



RFE Algorithm: Linear Regression

Selected Features:
['mba_p', 'specialisation_Mkt&HR', 'status_Placed']

RFE Algorithm: SVM Linear

Selected Features:
['ssc_p', 'hsc_p', 'status_Placed']

RFE Algorithm: Decision Tree

Selected Features:
['ssc_p', 'mba_p', 'status_Placed']

RFE Algorithm: Random Forest

Selected Features:
['ssc_p', 'hsc_p', 'status_Placed']

SELECTED FEATURES COMPARISON
    RFE Algorithm                             Selected Features
Linear Regression [mba_p, specialisation_Mkt&HR, status_Placed]
       SVM Linear                 [ssc_p, hsc_p, status_Placed]
    Decision Tree                 [ssc_p, mba_p, status_Placed]
    Random Forest                 [ssc_p, hsc_p, status_Placed]

ALL ALGORITHM COMPARISON
        RFE Algorithm Regression Algorithm  R2 Score           MSE
0   Linear Regression    Linear Regression  0.844674  2.750326e+09
1   Linear Regression           SVM Linear -0.237211  2.190699e+10
2   Linear Regression       SVM Non-Linear -0.239329  2.194450